In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from vetuner.drive_cycle import generate_drive_cycle
from vetuner.ve_surface import DEFAULT_MAP_AXIS, DEFAULT_RPM_AXIS

cycle = generate_drive_cycle()
print(f"{len(cycle)} samples over {cycle.time_s[-1]:.1f} s")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)

axes[0].plot(cycle.time_s, cycle.rpm, lw=1.0)
axes[0].set_ylabel("engine speed (rpm)")

axes[1].plot(cycle.time_s, cycle.tps * 100, lw=1.0, color="tab:orange")
axes[1].set_ylabel("throttle (%)")

axes[2].plot(cycle.time_s, cycle.map_kpa, lw=1.0, color="tab:green")
axes[2].set_ylabel("MAP (kPa abs)")
axes[2].set_xlabel("time (s)")

for ax in axes:
    ax.grid(alpha=0.3)

fig.suptitle("Synthetic drive cycle")
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(
    cycle.rpm,
    cycle.map_kpa,
    c=cycle.tps * 100,
    s=6,
    alpha=0.5,
    cmap="viridis",
)

for rpm_break in DEFAULT_RPM_AXIS:
    ax.axvline(rpm_break, color="grey", lw=0.4, alpha=0.5)
for map_break in DEFAULT_MAP_AXIS:
    ax.axhline(map_break, color="grey", lw=0.4, alpha=0.5)

ax.set_xlabel("engine speed (rpm)")
ax.set_ylabel("manifold pressure (kPa abs)")
ax.set_title("Operating point coverage against the VE table grid")
fig.colorbar(scatter, label="throttle (%)")
fig.tight_layout()

In [ ]:
rpm_bin = np.clip(np.digitize(cycle.rpm, DEFAULT_RPM_AXIS) - 1, 0, len(DEFAULT_RPM_AXIS) - 1)
map_bin = np.clip(np.digitize(cycle.map_kpa, DEFAULT_MAP_AXIS) - 1, 0, len(DEFAULT_MAP_AXIS) - 1)

counts = np.zeros((len(DEFAULT_RPM_AXIS), len(DEFAULT_MAP_AXIS)), dtype=int)
np.add.at(counts, (rpm_bin, map_bin), 1)

visited = (counts > 0).sum()
print(f"{visited} of {counts.size} cells visited ({100 * visited / counts.size:.0f}%)")
print(f"cells with fewer than 10 samples: {((counts > 0) & (counts < 10)).sum()}")

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(
    counts.T,
    origin="lower",
    aspect="auto",
    extent=[DEFAULT_RPM_AXIS[0], DEFAULT_RPM_AXIS[-1],
            DEFAULT_MAP_AXIS[0], DEFAULT_MAP_AXIS[-1]],
    cmap="magma",
)
ax.set_xlabel("engine speed (rpm)")
ax.set_ylabel("manifold pressure (kPa abs)")
ax.set_title("Samples per table cell")
fig.colorbar(im, label="sample count")